<a href="https://colab.research.google.com/github/mohanbaskaran7373-jpg/fraud-detection/blob/main/fraud_detection_using_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, euclidean_distances, adjusted_rand_score
from sklearn.base import BaseEstimator, ClusterMixin
import warnings
warnings.filterwarnings('ignore')

class ProductionKMeans(BaseEstimator, ClusterMixin):
    """Production-ready K-Means with all best practices"""

    def __init__(self, n_clusters=5, max_iter=300, random_state=42,
                 init='kmeans++', tol=1e-6, n_init=10, verbose=0):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.init = init
        self.tol = tol
        self.n_init = n_init  # Multiple runs like sklearn
        self.verbose = verbose

    def fit(self, X, y=None):
        """Fit with multiple initializations, return best by inertia"""
        best_inertia = float('inf')
        best_labels = None
        best_centroids = None
        best_history = None
        best_iter = 0

        for init_run in range(self.n_init):
            if self.verbose > 0:
                print(f"Init run {init_run+1}/{self.n_init}")

            # Single run
            km_single = self._fit_single(X)

            if km_single['inertia'] < best_inertia:
                best_inertia = km_single['inertia']
                best_labels = km_single['labels']
                best_centroids = km_single['centroids']
                best_history = km_single['inertia_history']
                best_iter = km_single['n_iter']

        # Store best results
        self.labels_ = best_labels
        self.centroids_ = best_centroids
        self.inertia_ = best_inertia
        self.n_iter_ = best_iter
        self.inertia_history_ = np.array(best_history)

        return self

    def _fit_single(self, X):
        """Single K-Means run with improved logic"""
        np.random.seed(self.random_state)

        # K-Means++ initialization (FIXED)
        centroids = self._kmeans_plus_plus_init(X)
        prev_inertia = float('inf')
        inertia_history = []

        for i in range(self.max_iter):
            # E-step: Assign clusters
            distances = euclidean_distances(X, centroids)
            labels = np.argmin(distances, axis=1)

            # M-step: Update centroids (handle empty clusters)
            new_centroids = np.zeros_like(centroids)
            cluster_counts = np.zeros(self.n_clusters)

            for k in range(self.n_clusters):
                mask = labels == k
                cluster_counts[k] = np.sum(mask)
                if cluster_counts[k] > 0:
                    new_centroids[k] = X[mask].mean(axis=0)
                else:
                    # Assign closest unassigned point
                    unassigned_mask = cluster_counts == 0
                    if np.any(unassigned_mask):
                        distances_to_empty = euclidean_distances(
                            X, new_centroids[~unassigned_mask]
                        ) if np.any(~unassigned_mask) else np.zeros((X.shape[0], 1))
                        new_centroids[k] = X[np.argmin(np.min(distances_to_empty, axis=1))]

            # Inertia calculation (CORRECTED)
            current_inertia = np.sum((X - new_centroids[labels])**2)
            inertia_history.append(current_inertia)

            # Convergence check
            if abs(prev_inertia - current_inertia) < self.tol:
                if self.verbose > 0:
                    print(f"✓ Converged at iter {i+1}")
                break

            prev_inertia = current_inertia
            centroids = new_centroids

        return {
            'labels': labels,
            'centroids': centroids,
            'inertia': current_inertia,
            'inertia_history': inertia_history,
            'n_iter': i + 1
        }

    def predict(self, X):
        """Predict cluster for new data"""
        distances = euclidean_distances(X, self.centroids_)
        return np.argmin(distances, axis=1)

    def _kmeans_plus_plus_init(self, X):
        """FIXED K-Means++ initialization"""
        n_samples, n_features = X.shape

        # Choose first centroid randomly
        centroid_idx = np.random.randint(0, n_samples)
        centroids = [X[centroid_idx]]

        # Choose remaining centroids
        for _ in range(1, self.n_clusters):
            dists = np.full(n_samples, float('inf'))
            for c in centroids:
                dists = np.minimum(dists, euclidean_distances(X, [c]).ravel())

            probs = dists / dists.sum()
            next_idx = np.random.choice(n_samples, p=probs)
            centroids.append(X[next_idx])

        return np.array(centroids)

# PRODUCTION EVALUATION PIPELINE
def evaluate_kmeans_production(X_scaled, true_labels=None):
    """Complete production evaluation"""
    results = []

    for k in range(2, 11):
        kmeans = ProductionKMeans(n_clusters=k, n_init=3, verbose=0)
        kmeans.fit(X_scaled)

        row = {
            'k': k,
            'inertia': kmeans.inertia_,
            'silhouette': silhouette_score(X_scaled, kmeans.labels_)
        }

        if true_labels is not None:
            row['ARI'] = adjusted_rand_score(true_labels, kmeans.labels_)

        results.append(row)

    df_results = pd.DataFrame(results)
    best_k = df_results.loc[df_results['silhouette'].idxmax(), 'k']

    return df_results, best_k, kmeans

# USAGE
np.random.seed(42)
# ... [your data generation code] ...

print("Production K-Means Results:")
results_df, best_k, final_model = evaluate_kmeans_production(X_scaled, true_labels)
print(results_df.round(4))
print(f"\n🏆 Optimal k: {best_k}")

Production K-Means Results:
    k   inertia  silhouette     ARI
0   2  141.6950      0.8364  1.0000
1   3  130.0654      0.4965  0.7586
2   4  119.0621      0.1547  0.5079
3   5  112.2835      0.1503  0.4244
4   6  106.3974      0.1348  0.3343
5   7  101.0629      0.1345  0.2923
6   8   98.9923      0.1257  0.2723
7   9   92.5474      0.1410  0.2498
8  10   90.3598      0.1393  0.2395

🏆 Optimal k: 2
